# AURA outfit-compatibility model — trained from scratch on Polyvore Outfits

This notebook trains AURA's **own** outfit-compatibility transformer. It learns from ~68,000 outfits that real people composed on Polyvore, so it judges whether *specific garments* (this navy blazer with these grey trousers) look good together — not just whether "a top goes with a bottom".

**How it works**
1. Every garment image is turned into a 512-number visual fingerprint by AURA's own garment CNN (`fashion_cnn_final.keras`, layer `fashion_feature_normalization`). The backend computes the exact same fingerprint for every photo a user uploads.
2. A 4-layer transformer (≈2.3M parameters, random initialisation) reads an outfit as a set of fingerprints + AURA categories and predicts *compatible / not compatible*.
3. It is evaluated on the two standard Polyvore Outfits benchmarks: **compatibility AUC** and **fill-in-the-blank (FITB) accuracy**, on both the *nondisjoint* and the harder *disjoint* split, against two baselines.

**Progress bars.** Every long step (unzip, fingerprint shards, each training epoch, validation scoring) shows a tqdm progress bar.

**Safe to re-run.** Fingerprints, per-epoch checkpoints and each finished model are saved to Drive immediately. If Colab disconnects or hits its GPU limit, open the notebook again and choose Run all: finished work is skipped and training resumes from the last epoch.

**Before you run**
- Open this notebook in **one tab only**. A second tab (or re-uploading the file while it is open) causes Colab's "file was updated remotely" save conflict.
- No GPU quota left? It still runs on CPU: cached fingerprints are reused and training resumes from the last saved epoch, just more slowly.
- Runtime → Change runtime type → **T4 GPU**.
- Copy `Capstone_Agents/model_artifacts/fashion_cnn_final.keras` and `fashion_cnn.metadata.json` into Google Drive at `MyDrive/AURA/aura_models/`.
- Then **Runtime → Run all**. Expected time on a free T4: download ≈10–15 min, fingerprints ≈20–40 min (cached to Drive, so a disconnect does not repeat finished shards), training ≈10–20 min per split.

**Outputs** (in `MyDrive/AURA/outfit_compat/`): `outfit_compat.weights.h5`, `outfit_compat.config.json` (architecture + held-out metrics), `evaluation.json`, `fitb_examples.png`, and a `disjoint/` copy used only for reporting.

**Dataset & licence.** Polyvore Outfits — Vasileva et al., *Learning Type-Aware Embeddings for Fashion Compatibility*, ECCV 2018 — distributed under CC BY 4.0; the authors' Hugging Face page asks that it be used for academic, non-profit research. This notebook downloads an unofficial mirror (`Stylique/Polyvore`) that keeps the original file layout and official splits. Cite the paper in your report.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, math, random, zipfile
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm   # preinstalled in Colab

AURA = Path('/content/drive/MyDrive/AURA')
CNN_DIR = AURA / 'aura_models'          # fashion_cnn_final.keras + fashion_cnn.metadata.json
OUT = AURA / 'outfit_compat'            # exported model + metrics
CACHE = AURA / 'polyvore_cache'         # garment fingerprints (reused by the LLM notebook)
DATA = Path('/content/polyvore')        # dataset on the fast local disk
SPLITS = ['nondisjoint', 'disjoint']    # nondisjoint model is deployed; disjoint is the harder benchmark
DEPLOY_SPLIT = 'nondisjoint'
POLYVORE_REPO = 'Stylique/Polyvore'     # mirror with the original Vasileva et al. layout
SEED = 42
for path in (OUT, CACHE, DATA):
    path.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

assert (CNN_DIR / 'fashion_cnn_final.keras').exists(), (
    'Copy Capstone_Agents/model_artifacts/fashion_cnn_final.keras to MyDrive/AURA/aura_models/ first.')

import tensorflow as tf
import keras
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('Cached fingerprint shards in Drive:', len(list(CACHE.glob('embeddings_*.npz'))))
if not tf.config.list_physical_devices('GPU'):
    print('WARNING: no GPU. Cached fingerprints are reused, but computing missing ones and training are much slower on CPU.')
os.makedirs('/content/aura_ml', exist_ok=True)
open('/content/aura_ml/__init__.py', 'w').close()
sys.path.insert(0, '/content')


## Shared model code
The next cell writes `ml/outfit_compat.py` from the backend **byte-for-byte**. A backend test checks that this cell and the repository file are identical, so the trained weights always load into the same architecture.


In [ ]:
%%writefile aura_ml/outfit_compat.py
"""AURA outfit-compatibility transformer (trained from scratch on Polyvore Outfits).

Each garment is represented by the 512-d ``fashion_feature_normalization``
embedding from AURA's own EfficientNetV2 garment CNN plus its broad AURA
category. A small transformer encoder reads the whole outfit as a set and
predicts whether the pieces belong together.

This file is shared verbatim by the backend and
``training/AURA_Outfit_Compatibility_Polyvore_Colab.ipynb``. Keep it free of
backend imports so it can run unchanged in Colab.
"""
from __future__ import annotations

from typing import Any, Sequence

import numpy as np

EMBEDDING_DIM = 512
MAX_ITEMS = 16
EMBEDDING_LAYER = "fashion_feature_normalization"

# 0 is padding, the CLS token uses its own learned vector.
AURA_CATEGORIES = ("Top", "Bottom", "Dress", "Footwear", "Outerwear", "Accessory")
CATEGORY_TO_ID = {name: index + 1 for index, name in enumerate(AURA_CATEGORIES)}

# Polyvore Outfits "semantic_category" -> AURA wardrobe category.
POLYVORE_TO_AURA = {
    "tops": "Top",
    "bottoms": "Bottom",
    "all-body": "Dress",
    "shoes": "Footwear",
    "outerwear": "Outerwear",
    "bags": "Accessory",
    "accessories": "Accessory",
    "jewellery": "Accessory",
    "hats": "Accessory",
    "scarves": "Accessory",
    "sunglasses": "Accessory",
}

DEFAULT_CONFIG: dict[str, Any] = {
    "embedding_dim": EMBEDDING_DIM,
    "max_items": MAX_ITEMS,
    "model_dim": 256,
    "num_heads": 4,
    "num_layers": 4,
    "ff_dim": 512,
    "dropout": 0.1,
    "input_noise": 0.05,
    "feature_dropout": 0.1,
}


def category_id(category: str | None) -> int:
    """Map an AURA category name to its integer id (unknown -> Accessory)."""
    return CATEGORY_TO_ID.get(str(category or ""), CATEGORY_TO_ID["Accessory"])


def encode_outfits(
    outfits: Sequence[Sequence[tuple[np.ndarray, str]]],
    max_items: int = MAX_ITEMS,
) -> tuple[np.ndarray, np.ndarray]:
    """Pack outfits of (embedding, AURA category) pairs into padded arrays."""
    features = np.zeros((len(outfits), max_items, EMBEDDING_DIM), dtype=np.float32)
    categories = np.zeros((len(outfits), max_items), dtype=np.int32)
    for row, outfit in enumerate(outfits):
        for column, (embedding, category) in enumerate(list(outfit)[:max_items]):
            features[row, column] = np.asarray(embedding, dtype=np.float32)
            categories[row, column] = category_id(category)
    return features, categories


def build_model(config: dict[str, Any] | None = None):
    """Build the Keras model. Weights are random until trained or loaded."""
    import keras
    from keras import layers, ops

    cfg = {**DEFAULT_CONFIG, **(config or {})}
    d_model = int(cfg["model_dim"])

    @keras.saving.register_keras_serializable(package="AURAOutfitCompat")
    class OutfitEncoder(layers.Layer):
        def __init__(self, cfg: dict[str, Any], **kwargs: Any) -> None:
            super().__init__(**kwargs)
            self.cfg = dict(cfg)
            self.project = layers.Dense(d_model, name="item_projection")
            self.category_embedding = layers.Embedding(
                len(AURA_CATEGORIES) + 1, d_model, name="category_embedding"
            )
            self.noise = layers.GaussianNoise(float(cfg["input_noise"]))
            self.feature_dropout = layers.Dropout(float(cfg["feature_dropout"]))
            # Keras weight saving only follows layers held directly or in flat lists, so each
            # sub-layer type gets its own list (a list of dicts silently left the blocks out of
            # .weights.h5). Creation order and layer names match the original list-of-dicts version.
            self.block_norm1, self.block_attention, self.block_drop1 = [], [], []
            self.block_norm2, self.block_ff1, self.block_ff2, self.block_drop2 = [], [], [], []
            for index in range(int(cfg["num_layers"])):
                self.block_norm1.append(layers.LayerNormalization(epsilon=1e-6, name=f"block{index}_norm1"))
                self.block_attention.append(layers.MultiHeadAttention(
                    num_heads=int(cfg["num_heads"]),
                    key_dim=d_model // int(cfg["num_heads"]),
                    dropout=float(cfg["dropout"]),
                    name=f"block{index}_attention",
                ))
                self.block_drop1.append(layers.Dropout(float(cfg["dropout"])))
                self.block_norm2.append(layers.LayerNormalization(epsilon=1e-6, name=f"block{index}_norm2"))
                self.block_ff1.append(layers.Dense(int(cfg["ff_dim"]), activation="gelu", name=f"block{index}_ff1"))
                self.block_ff2.append(layers.Dense(d_model, name=f"block{index}_ff2"))
                self.block_drop2.append(layers.Dropout(float(cfg["dropout"])))
            self.final_norm = layers.LayerNormalization(epsilon=1e-6, name="final_norm")

        def build(self, input_shape: Any) -> None:
            self.cls_token = self.add_weight(
                name="cls_token", shape=(1, 1, d_model), initializer="random_normal", trainable=True
            )
            super().build(input_shape)

        def call(self, features: Any, categories: Any, training: bool | None = None) -> Any:
            features = self.noise(features, training=training)
            features = self.feature_dropout(features, training=training)
            tokens = self.project(features) + self.category_embedding(categories)
            batch = ops.shape(tokens)[0]
            cls = ops.broadcast_to(self.cls_token, (batch, 1, d_model))
            tokens = ops.concatenate([cls, tokens], axis=1)
            valid = ops.concatenate(
                [ops.ones((batch, 1), dtype="bool"), ops.greater(categories, 0)], axis=1
            )
            mask = ops.logical_and(ops.expand_dims(valid, 1), ops.expand_dims(valid, 2))
            x = tokens
            for index in range(len(self.block_attention)):
                y = self.block_norm1[index](x)
                y = self.block_attention[index](y, y, attention_mask=mask, training=training)
                x = x + self.block_drop1[index](y, training=training)
                y = self.block_norm2[index](x)
                y = self.block_ff2[index](self.block_ff1[index](y))
                x = x + self.block_drop2[index](y, training=training)
            return self.final_norm(x)[:, 0, :]

        def get_config(self) -> dict[str, Any]:
            return {**super().get_config(), "cfg": self.cfg}

    features = keras.Input((int(cfg["max_items"]), int(cfg["embedding_dim"])), name="features")
    categories = keras.Input((int(cfg["max_items"]),), dtype="int32", name="categories")
    pooled = OutfitEncoder(cfg, name="outfit_encoder")(features, categories)
    hidden = layers.Dense(d_model, activation="gelu", name="head_hidden")(pooled)
    hidden = layers.Dropout(float(cfg["dropout"]))(hidden)
    logit = layers.Dense(1, name="compatibility_logit")(hidden)
    return keras.Model({"features": features, "categories": categories}, logit, name="aura_outfit_compat")


def predict_scores(model: Any, features: np.ndarray, categories: np.ndarray, batch_size: int = 512) -> np.ndarray:
    """Return compatibility probabilities in [0, 1] for packed outfits."""
    logits = model.predict(
        {"features": features, "categories": categories}, batch_size=batch_size, verbose=0
    )
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    return 1.0 / (1.0 + np.exp(-logits))


## 1. Download Polyvore Outfits

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    POLYVORE_REPO, repo_type='dataset', local_dir=str(DATA),
    allow_patterns=['*.json', '*.txt', 'categories.csv', 'images.zip'],
    ignore_patterns=['*hglmm*', 'maryland_polyvore_hardneg/*'],
)
if not any(DATA.rglob('*.jpg')):
    with zipfile.ZipFile(DATA / 'images.zip') as archive:
        for member in tqdm(archive.infolist(), desc='Unzipping images (~2.5 GB)', unit='file'):
            archive.extract(member, DATA)
first_image = next(DATA.rglob('*.jpg'))
IMAGE_DIR = first_image.parent
print('Images in', IMAGE_DIR, '| example', first_image.name)
for split in SPLITS:
    for name in ['train.json', 'valid.json', 'test.json', 'compatibility_valid.txt', 'compatibility_test.txt',
                 'fill_in_blank_valid.json', 'fill_in_blank_test.json']:
        assert (DATA / split / name).exists(), f'missing {split}/{name}'


## 2. Metadata, categories and official splits

In [ ]:
from aura_ml.outfit_compat import (POLYVORE_TO_AURA, AURA_CATEGORIES, MAX_ITEMS, EMBEDDING_LAYER,
                                   build_model, predict_scores, CATEGORY_TO_ID)

item_meta = json.load(open(DATA / 'polyvore_item_metadata.json'))
category_rows = {}
for line in open(DATA / 'categories.csv', encoding='utf-8'):
    parts = [part.strip() for part in line.strip().split(',')]
    if len(parts) >= 3 and parts[0].isdigit():
        category_rows[parts[0]] = (parts[1], parts[2])

def semantic_category(item_id):
    meta = item_meta.get(item_id, {})
    semantic = meta.get('semantic_category')
    if not semantic:
        semantic = category_rows.get(str(meta.get('category_id')), (None, None))[1]
    return semantic

def aura_category(item_id):
    return POLYVORE_TO_AURA.get(semantic_category(item_id), 'Accessory')

def load_split(split):
    # part -> {"setid_index": item_id}, part -> [outfit item lists]
    lookup, outfits = {}, {}
    for part in ['train', 'valid', 'test']:
        lookup[part], outfits[part] = {}, []
        for outfit in json.load(open(DATA / split / f'{part}.json')):
            ordered = sorted(outfit['items'], key=lambda entry: entry['index'])
            for entry in ordered:
                lookup[part][f"{outfit['set_id']}_{entry['index']}"] = entry['item_id']
            outfits[part].append([entry['item_id'] for entry in ordered])
    def compat(part):
        rows = []
        for line in open(DATA / split / f'compatibility_{part}.txt'):
            tokens = line.split()
            if tokens:
                rows.append((int(tokens[0]), [lookup[part][token] for token in tokens[1:] if token in lookup[part]]))
        return rows
    def fitb(part):
        rows = []
        for question in json.load(open(DATA / split / f'fill_in_blank_{part}.json')):
            set_id = question['question'][0].split('_')[0]
            answers = question['answers']
            correct = next((i for i, token in enumerate(answers) if token.split('_')[0] == set_id), 0)
            rows.append(([lookup[part][t] for t in question['question'] if t in lookup[part]],
                         [lookup[part].get(t) for t in answers], correct))
        return rows
    return {'outfits': outfits, 'compat': {p: compat(p) for p in ['valid', 'test']},
            'fitb': {p: fitb(p) for p in ['valid', 'test']}}

splits = {split: load_split(split) for split in tqdm(SPLITS, desc='Loading splits', unit='split')}
needed = set()
for data in splits.values():
    for part_outfits in data['outfits'].values():
        for outfit in part_outfits: needed.update(outfit)
    for rows in data['fitb'].values():
        for question, answers, _ in rows: needed.update(question); needed.update(a for a in answers if a)
needed = sorted(item for item in needed if (IMAGE_DIR / f'{item}.jpg').exists())
for split, data in splits.items():
    print(split, {part: len(rows) for part, rows in data['outfits'].items()},
          '| compat test', len(data['compat']['test']), '| FITB test', len(data['fitb']['test']))
print('Garments with images to fingerprint:', len(needed))
from collections import Counter
print('AURA category mix:', Counter(aura_category(item) for item in needed[:20000]))


## 3. Garment fingerprints with AURA's CNN (cached to Drive)

In [ ]:
# Fingerprint every garment with AURA's own CNN (same layer + preprocessing as the backend).
# Valid shards already in Drive are reused; the CNN is only loaded if a shard has to be computed.
from aura_ml.outfit_compat import EMBEDDING_DIM

IMAGE_SIZE = json.load(open(CNN_DIR / 'fashion_cnn.metadata.json'))['image_size']
SHARD, IMAGE_BATCH = 25000, 256
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        cnn = tf.keras.models.load_model(CNN_DIR / 'fashion_cnn_final.keras', compile=False)
        embedder = tf.keras.Model(cnn.input, cnn.get_layer(EMBEDDING_LAYER).output)
    return embedder

def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    # Backend: PIL LANCZOS resize to 224x224 without padding, pixels in 0-255.
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE), method='lanczos3', antialias=True)
    return tf.clip_by_value(image, 0.0, 255.0)

def read_shard(shard_path, ids):
    """Cached vectors if the Drive shard is complete and holds exactly `ids`, else None.
    A shard that does not match is never deleted: it is renamed to *.bad_<time> and recomputed."""
    if not shard_path.exists():
        return None
    stored_ids, vectors = None, None
    try:
        with np.load(shard_path) as shard:
            stored_ids, vectors = shard['ids'].tolist(), shard['vectors']
    except Exception as error:
        tqdm.write(f'{shard_path.name}: unreadable ({error!r})')
    if (stored_ids == ids and vectors.shape == (len(ids), EMBEDDING_DIM)
            and np.isfinite(vectors).all()):
        return vectors
    backup = shard_path.with_name(f'{shard_path.stem}.bad_{int(time.time())}')
    shard_path.rename(backup)
    tqdm.write(f'{shard_path.name}: does not match the expected garments -> kept as {backup.name}, recomputing')
    return None

ids, vectors = [], []
shard_bar = tqdm(range(0, len(needed), SHARD), desc='Fingerprint shards', unit='shard')
for start in shard_bar:
    number = start // SHARD
    shard_path = CACHE / f'embeddings_{number:03d}.npz'
    shard_ids = needed[start:start + SHARD]
    shard_vectors = read_shard(shard_path, shard_ids)
    if shard_vectors is None:
        model_ = get_embedder()
        paths = [str(IMAGE_DIR / f'{item}.jpg') for item in shard_ids]
        dataset = (tf.data.Dataset.from_tensor_slices(paths)
                   .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
                   .batch(IMAGE_BATCH).prefetch(tf.data.AUTOTUNE))
        began = time.time()
        parts = [np.asarray(model_.predict_on_batch(batch))
                 for batch in tqdm(dataset, total=math.ceil(len(shard_ids) / IMAGE_BATCH),
                                   desc=f'  shard {number:03d}', unit='batch', leave=False)]
        shard_vectors = np.concatenate(parts).astype(np.float16)
        # Write under a temporary name, then rename, so a disconnect never leaves a half-written shard.
        partial = CACHE / f'partial_{shard_path.name}'
        np.savez(partial, ids=np.array(shard_ids), vectors=shard_vectors)
        os.replace(partial, shard_path)
        tqdm.write(f'shard {number:03d}: {len(shard_ids)} garments in {time.time() - began:.0f}s')
        shard_bar.set_postfix_str(f'{shard_path.name} computed')
    else:
        shard_bar.set_postfix_str(f'{shard_path.name} cached')
    ids.extend(shard_ids); vectors.append(shard_vectors)

EMB = np.concatenate(vectors).astype(np.float32)
del vectors
INDEX = {item: position for position, item in enumerate(ids)}
CAT = np.array([CATEGORY_TO_ID[aura_category(item)] for item in ids], dtype=np.int32)
print('Fingerprints:', EMB.shape)


## 4. Training sampler, baselines and benchmark evaluation

In [ ]:
rng = np.random.default_rng(SEED)

def to_indices(outfit):
    return [INDEX[item] for item in outfit if item in INDEX][:MAX_ITEMS]

def pack(index_lists):
    batch = np.full((len(index_lists), MAX_ITEMS), -1, dtype=np.int64)
    for row, indices in enumerate(index_lists):
        batch[row, :len(indices)] = indices[:MAX_ITEMS]
    valid = batch >= 0
    features = EMB[np.where(valid, batch, 0)] * valid[..., None]
    categories = np.where(valid, CAT[np.where(valid, batch, 0)], 0).astype(np.int32)
    return {'features': features.astype(np.float32), 'categories': categories}

class OutfitSampler:
    # Positives: real outfits (sometimes with one accessory dropped, like a sparse user wardrobe).
    # Negatives: the same outfit with 1-2 garments swapped (hard) or every garment swapped (easy),
    # always replacing a garment with another garment of the same AURA category.
    def __init__(self, outfits):
        self.outfits = [o for o in (to_indices(outfit) for outfit in outfits) if len(o) >= 2]
        members = sorted({i for outfit in self.outfits for i in outfit})
        self.pool = {c: np.array([i for i in members if CAT[i] == c]) for c in range(1, len(AURA_CATEGORIES) + 1)}

    def _swap(self, outfit, positions):
        outfit = list(outfit)
        for position in positions:
            pool = self.pool.get(int(CAT[outfit[position]]))
            if pool is not None and len(pool):
                outfit[position] = int(rng.choice(pool))
        return outfit

    def batch(self, size=256):
        lists, labels = [], []
        for index in rng.integers(0, len(self.outfits), size // 2):
            outfit = list(self.outfits[index])
            accessories = [p for p, i in enumerate(outfit) if CAT[i] == CATEGORY_TO_ID['Accessory']]
            if len(outfit) > 3 and accessories and rng.random() < 0.3:
                outfit.pop(int(rng.choice(accessories)))
            lists.append(outfit); labels.append(1.0)
            if rng.random() < 0.6:
                k = 1 if len(outfit) < 4 or rng.random() < 0.6 else 2
                positions = rng.choice(len(outfit), size=k, replace=False)
            else:
                positions = range(len(outfit))
            lists.append(self._swap(outfit, positions)); labels.append(0.0)
        return pack(lists), np.array(labels, dtype=np.float32)

def score_lists(model, index_lists, batch_size=1024, desc='Scoring'):
    scores = []
    for start in tqdm(range(0, len(index_lists), batch_size), desc=desc, unit='batch', leave=False):
        packed = pack(index_lists[start:start + batch_size])
        scores.append(predict_scores(model, packed['features'], packed['categories']))
    return np.concatenate(scores) if scores else np.array([])

def cosine_baseline(index_lists):
    # Untrained baseline: mean pairwise cosine similarity of the CNN fingerprints.
    out = []
    for indices in tqdm(index_lists, desc='Cosine baseline', unit='outfit', leave=False):
        v = EMB[indices]; v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
        sim = v @ v.T; n = len(indices)
        out.append((sim.sum() - n) / max(1, n * (n - 1)))
    return np.array(out)

from sklearn.metrics import roc_auc_score

def evaluate(scorer, data, part):
    compat = [(label, to_indices(items)) for label, items in data['compat'][part]]
    compat = [(label, items) for label, items in compat if len(items) >= 2]
    scores = scorer([items for _, items in compat])
    auc = roc_auc_score([label for label, _ in compat], scores)
    # Score every fill-in-the-blank candidate in one batch (much faster than one call per question).
    lists, spans = [], []
    for question, answers, answer_index in data['fitb'][part]:
        q = to_indices(question)[:MAX_ITEMS - 1]  # leave room for the candidate garment
        if not q or None in answers or any(a not in INDEX for a in answers):
            continue
        spans.append((len(lists), len(answers), answer_index))
        lists += [q + [INDEX[a]] for a in answers]
    fitb_scores = scorer(lists) if lists else np.array([])
    correct = sum(int(np.argmax(fitb_scores[start:start + n]) == answer) for start, n, answer in spans)
    return {'compatibility_auc': float(auc), 'fitb_accuracy': correct / max(1, len(spans)),
            'compat_examples': len(compat), 'fitb_questions': len(spans)}
print('Samplers ready.')


## 4b. Estimate training time (optional, run before committing to a long session)
Times a few real training steps and one validation pass **on this machine** with a throwaway model, then extrapolates to the full run. Nothing is written to Drive and the training random stream is left untouched. Run it on Colab (GPU or CPU runtime) and locally, and compare the totals. Skip it once you have decided.

In [ ]:
# Measures this machine's real speed and extrapolates; uses a throwaway model, writes nothing.
WARMUP, TIMED = 5, 30                       # raise TIMED for a steadier estimate
EST_BATCH, EST_EPOCHS, EST_PATIENCE = 256, 30, 4   # same values as the training cell
device = 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'
rng_state = rng.bit_generator.state         # restore afterwards so training samples stay identical
low_total = high_total = 0.0
print(f'Timing on {device} ...')
for split in SPLITS:
    data = splits[split]
    sampler = OutfitSampler(data['outfits']['train'])
    steps = max(200, len(sampler.outfits) // (EST_BATCH // 2))
    model = build_model()
    model.compile(optimizer=keras.optimizers.AdamW(3e-4, weight_decay=0.01),
                  loss=keras.losses.BinaryCrossentropy(from_logits=True, label_smoothing=0.05))
    began = time.time()
    batches = [sampler.batch(EST_BATCH) for _ in tqdm(range(WARMUP + TIMED), desc=f'{split} sampling', leave=False)]
    sample_s = (time.time() - began) / (WARMUP + TIMED)
    for x, y in batches[:WARMUP]:            # first calls include graph compilation; not timed
        model.train_on_batch(x, y)
    began = time.time()
    for x, y in tqdm(batches[WARMUP:], desc=f'{split} train steps', leave=False):
        model.train_on_batch(x, y)
    train_s = (time.time() - began) / TIMED
    began = time.time()
    evaluate(lambda lists: score_lists(model, lists, desc=f'{split} valid pass'), data, 'valid')
    valid_s = time.time() - began
    epoch_s = steps * (sample_s + train_s) + valid_s
    fewest = EST_PATIENCE + 1                # earliest possible early stop
    low_total += fewest * epoch_s; high_total += EST_EPOCHS * epoch_s
    print(f'{split:12s} {steps} steps/epoch | train {train_s * 1000:.0f} ms/step + sampling {sample_s * 1000:.0f} ms/step '
          f'| validation {valid_s:.0f}s')
    print(f'{"":12s} = {epoch_s / 60:.1f} min/epoch -> {fewest * epoch_s / 3600:.2f} h (earliest early stop, {fewest} epochs) '
          f'to {EST_EPOCHS * epoch_s / 3600:.2f} h (all {EST_EPOCHS} epochs)')
    del model, batches
rng.bit_generator.state = rng_state
print(f'\nBoth splits on {device}: {low_total / 3600:.2f} h to {high_total / 3600:.2f} h '
      '(plus a few minutes for test scoring and export). Checkpoints are saved every epoch, so this can be split across sessions.')


## 5. Train from scratch (both splits), select on validation, evaluate on test

In [ ]:
from datetime import datetime, timezone
from aura_ml.outfit_compat import DEFAULT_CONFIG

EPOCHS, BATCH, PATIENCE = 30, 256, 4
CKPT = OUT / 'checkpoints'; CKPT.mkdir(parents=True, exist_ok=True)
EVAL_PATH = OUT / 'evaluation.json'
results = json.loads(EVAL_PATH.read_text()) if EVAL_PATH.exists() else {}
models = {}

class StepProgress(keras.callbacks.Callback):
    """tqdm bar over the training steps of one epoch (shows the running loss)."""
    def __init__(self, steps, desc, refresh_every=20):
        super().__init__(); self.steps, self.desc, self.refresh_every = steps, desc, refresh_every
    def on_epoch_begin(self, epoch, logs=None):
        self.bar = tqdm(total=self.steps, desc=self.desc, unit='step', leave=False)
    def on_train_batch_end(self, batch, logs=None):
        self.bar.update(1)
        if logs and 'loss' in logs and batch % self.refresh_every == 0:
            self.bar.set_postfix(loss=f"{float(logs['loss']):.4f}")
    def on_epoch_end(self, epoch, logs=None):
        self.bar.close()

def save_weights_safely(model, path):
    # Write to a temporary file, then rename: a disconnect mid-save cannot corrupt the checkpoint.
    partial = path.with_name('partial_' + path.name)
    model.save_weights(str(partial)); os.replace(partial, path)

def write_text_safely(path, content):
    partial = path.with_name('partial_' + path.name)
    partial.write_text(content); os.replace(partial, path)

print('Training device:', 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU (slower, but checkpoints still resume)')

def final_dir(split):
    return OUT if split == DEPLOY_SPLIT else OUT / split

def export(model, split):
    # Saved to Drive as soon as a split finishes, so a disconnect never loses a trained model.
    folder = final_dir(split); folder.mkdir(parents=True, exist_ok=True)
    save_weights_safely(model, folder / 'outfit_compat.weights.h5')
    config = {
        'model_config': DEFAULT_CONFIG,
        'training_split': f'Polyvore Outfits {split} (train part only)',
        'test_metrics': results[split]['test'],
        'baselines': {'cosine_similarity': results[split]['baseline_cosine_similarity'], 'random': results[split]['baseline_random']},
        'embedding_source': 'AURA fashion_cnn_final.keras, layer fashion_feature_normalization (512-d)',
        'image_preprocessing': 'RGB, resized to 224x224 (Lanczos), pixel range 0-255',
        'score_meaning': 'sigmoid probability that the garments form a compatible outfit (balanced training); not calibrated for personal taste',
        'dataset': 'Polyvore Outfits (Vasileva et al., ECCV 2018), CC BY 4.0',
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'tensorflow': tf.__version__, 'keras': keras.__version__,
    }
    write_text_safely(folder / 'outfit_compat.config.json', json.dumps(config, indent=2))
    write_text_safely(EVAL_PATH, json.dumps(results, indent=2))
    print('Saved', split, 'model to', folder)

for split in SPLITS:
    data = splits[split]
    model = build_model()
    done = final_dir(split) / 'outfit_compat.weights.h5'
    if done.exists() and split in results:
        model.load_weights(str(done)); models[split] = model
        print(f'== {split}: already trained, loaded from Drive | test', results[split]['test'])
        continue
    sampler = OutfitSampler(data['outfits']['train'])
    steps = max(200, len(sampler.outfits) // (BATCH // 2))
    keras.utils.set_random_seed(SEED)
    best_path, last_path, state_path = (CKPT / f'{split}_best.weights.h5', CKPT / f'{split}_last.weights.h5',
                                        CKPT / f'{split}_state.json')
    first, best, waited, history = 1, -1.0, 0, []
    if state_path.exists() and last_path.exists() and best_path.exists():
        state = json.loads(state_path.read_text())
        model.load_weights(str(last_path))
        first, best, waited, history = state['epoch'] + 1, state['best'], state['waited'], state['history']
        print(f'== {split}: resuming after epoch {state["epoch"]} (best valid score {best:.4f})')
    remaining = max(1, EPOCHS - first + 1)
    start_lr = 3e-4 * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (first - 1) / EPOCHS)))
    schedule = keras.optimizers.schedules.CosineDecay(start_lr, decay_steps=remaining * steps, alpha=0.05)
    model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=0.01),
                  loss=keras.losses.BinaryCrossentropy(from_logits=True, label_smoothing=0.05))
    def generator():
        while True:
            yield sampler.batch(BATCH)
    signature = ({'features': tf.TensorSpec((None, MAX_ITEMS, 512), tf.float32),
                  'categories': tf.TensorSpec((None, MAX_ITEMS), tf.int32)},
                 tf.TensorSpec((None,), tf.float32))
    stream = tf.data.Dataset.from_generator(generator, output_signature=signature).prefetch(4)
    print(f'== {split}: {len(sampler.outfits)} training outfits, {steps} steps/epoch, {model.count_params():,} parameters')
    epoch_bar = tqdm(range(first, EPOCHS + 1), initial=first - 1, total=EPOCHS, desc=f'{split} epochs', unit='epoch')
    for epoch in epoch_bar:
        if waited >= PATIENCE:
            break
        began = time.time()
        loss = model.fit(stream, steps_per_epoch=steps, epochs=1, verbose=0,
                         callbacks=[StepProgress(steps, f'  epoch {epoch} train')]).history['loss'][-1]
        valid = evaluate(lambda lists: score_lists(model, lists, desc=f'  epoch {epoch} valid'), data, 'valid')
        selection = valid['compatibility_auc'] + valid['fitb_accuracy']
        history.append({'epoch': epoch, 'loss': float(loss), **valid})
        if selection > best:
            best, waited = selection, 0
            save_weights_safely(model, best_path)
        else:
            waited += 1
        save_weights_safely(model, last_path)
        write_text_safely(state_path, json.dumps({'epoch': epoch, 'best': best, 'waited': waited, 'history': history}))
        epoch_bar.set_postfix(loss=f'{loss:.4f}', auc=f"{valid['compatibility_auc']:.4f}",
                              fitb=f"{valid['fitb_accuracy']:.4f}", patience=f'{waited}/{PATIENCE}')
        tqdm.write(f"{split} epoch {epoch:2d} loss {loss:.4f} | valid AUC {valid['compatibility_auc']:.4f} "
                   f"FITB {valid['fitb_accuracy']:.4f} | {time.time() - began:.0f}s | checkpoint saved")
    epoch_bar.close()
    if waited >= PATIENCE:
        print('Early stopping.')
    model.load_weights(str(best_path))
    test = evaluate(lambda lists: score_lists(model, lists, desc='Test scoring'), data, 'test')
    cosine = evaluate(cosine_baseline, data, 'test')
    results[split] = {'test': test, 'baseline_cosine_similarity': cosine,
                      'baseline_random': {'compatibility_auc': 0.5, 'fitb_accuracy': 0.25},
                      'history': history, 'training_outfits': len(sampler.outfits)}
    models[split] = model
    print(split, 'TEST', test, '| cosine baseline', {k: round(v, 4) for k, v in cosine.items() if 'examples' not in k and 'questions' not in k})
    export(model, split)


## 6. Results

In [ ]:
print(f"{'split':12s} {'model':28s} {'AUC':>7s} {'FITB':>7s}")
for split, result in results.items():
    for name, metrics in [('AURA transformer (trained)', result['test']),
                          ('CNN cosine similarity', result['baseline_cosine_similarity']),
                          ('random', result['baseline_random'])]:
        print(f"{split:12s} {name:28s} {metrics['compatibility_auc']:7.4f} {metrics['fitb_accuracy']:7.4f}")
print('\nPublished reference points on Polyvore Outfits (different, end-to-end trained image encoders):')
print('  Type-aware embeddings (Vasileva et al. 2018) and later transformer models report higher numbers;')
print('  quote them from the papers you cite, and state that AURA keeps its own frozen CNN for deployment.')


## 7. Export for the backend

In [ ]:
# Each split was already saved to Drive right after training; this re-checks the deployed one.
check = build_model(); check.load_weights(str(OUT / 'outfit_compat.weights.h5'))
probe = pack([to_indices(splits[DEPLOY_SPLIT]['outfits']['test'][0])])
assert abs(float(predict_scores(check, **probe)[0]) - float(predict_scores(models[DEPLOY_SPLIT], **probe)[0])) < 1e-5
print('Deployed model OK in', OUT, '| files:', sorted(p.name for p in OUT.glob('outfit_compat.*')))


## 8. Qualitative fill-in-the-blank examples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

data, model = splits[DEPLOY_SPLIT], models[DEPLOY_SPLIT]
questions = [q for q in data['fitb']['test'] if None not in q[1] and all(a in INDEX for a in q[1])][:5]
fig, axes = plt.subplots(len(questions), 9, figsize=(18, 2.3 * len(questions)))
for row, (question, answers, correct) in enumerate(questions):
    q = to_indices(question)
    scores = score_lists(model, [q + [INDEX[a]] for a in answers])
    pick = int(np.argmax(scores))
    tiles = question[:5] + [None] * (5 - len(question[:5])) + answers
    for col in range(9):
        axis = axes[row, col]; axis.axis('off')
        if col < 5 and tiles[col]:
            axis.imshow(Image.open(IMAGE_DIR / f'{tiles[col]}.jpg'))
        elif col >= 5:
            answer = answers[col - 5]
            axis.imshow(Image.open(IMAGE_DIR / f'{answer}.jpg'))
            mark = ('PICK ' if col - 5 == pick else '') + ('(correct)' if col - 5 == correct else '')
            axis.set_title(f'{scores[col - 5]:.2f} {mark}', fontsize=8, color='green' if col - 5 == correct else 'black')
plt.suptitle('FITB examples: outfit (left) and four candidate garments with model scores (right)')
plt.tight_layout(); plt.savefig(OUT / 'fitb_examples.png', dpi=110); plt.show()


## Next steps
1. Download the whole `MyDrive/AURA/outfit_compat/` folder and place `outfit_compat.weights.h5` and `outfit_compat.config.json` in `Capstone_Agents/model_artifacts/outfit_compat/`.
2. Restart the backend and open `GET /models/outfit-compatibility/status` — `available` should be `true` and the test metrics are shown.
3. Existing wardrobe items get their fingerprints automatically the first time recommendations are requested (computed from the saved photo); new uploads store them immediately.
4. Run `AURA_Stylist_LLM_LoRA_Colab.ipynb` next — it reuses the fingerprints in `MyDrive/AURA/polyvore_cache/` and this model.

Use `evaluation.json` and `fitb_examples.png` in your report. The disjoint-split numbers are the honest generalisation estimate because no test garment appears in training.
